# Sentence clustering of `sampled_autism.jsonl`
Load, split abstracts into sentences, embed with Qwen3-Embedding, agglomerative clustering.

In [ ]:
# !pip install -q pandas numpy nltk sentence-transformers scikit-learn scipy matplotlib

In [ ]:
import re
import numpy as np
import pandas as pd
import nltk

PATH = "sampled_autism.jsonl"
MODEL = "Qwen/Qwen3-Embedding-0.6B"   # or Qwen/Qwen3-Embedding-4B / -8B
BATCH_SIZE = 64
MAX_LENGTH = 256                       # tokens per sentence
N_CLUSTERS = 20                        # set to None to use DISTANCE_THRESHOLD instead
DISTANCE_THRESHOLD = None              # e.g. 1.0 (ward on normalized vectors)
LINKAGE = "ward"                       # ward | average | complete | single

## Load

In [ ]:
df = pd.read_json(PATH, lines=True)
print(df.shape)
print(df.columns.tolist())
df.head(3)

## Split abstracts into sentences

In [ ]:
for pkg in ("punkt_tab", "punkt"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

def split_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return []
    try:
        sents = nltk.sent_tokenize(text)
    except LookupError:                      # fallback if punkt is unavailable
        sents = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", text)
    return [s.strip() for s in sents if s.strip()]

df["sentences"] = df["abstract"].map(split_sentences)
sent = (df.reset_index().rename(columns={"index": "row"})
          .explode("sentences").dropna(subset=["sentences"])
          .rename(columns={"sentences": "sentence"}))
sent["sent_idx"] = sent.groupby("row").cumcount()
sent = sent.reset_index(drop=True)
print(f"{len(df):,} abstracts -> {len(sent):,} sentences")
sent[["row", "sent_idx", "sentence"]].head()

## Embed every sentence

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
model = SentenceTransformer(MODEL, device=device)
model.max_seq_length = MAX_LENGTH

emb = model.encode(sent["sentence"].tolist(), batch_size=BATCH_SIZE,
                   normalize_embeddings=True, show_progress_bar=True)
print(emb.shape)
np.save("sentence_embeddings.npy", emb)

## Agglomerative clustering
Agglomerative clustering builds the full pairwise-distance matrix, so memory grows with the square of the number of sentences.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

n = len(emb)
print(f"{n:,} sentences -> distance matrix ~{n * n * 8 / 1e9:.1f} GB")

kw = dict(linkage=LINKAGE)
if N_CLUSTERS is not None:
    kw["n_clusters"] = N_CLUSTERS
else:
    kw.update(n_clusters=None, distance_threshold=DISTANCE_THRESHOLD)
if LINKAGE != "ward":
    kw["metric"] = "cosine"

agg = AgglomerativeClustering(**kw)
sent["cluster"] = agg.fit_predict(emb)
sent["cluster"].value_counts().sort_index()

## Inspect clusters

In [ ]:
centroids = {c: emb[sent.index[sent["cluster"] == c]].mean(0) for c in sorted(sent["cluster"].unique())}
for c, cen in centroids.items():
    idx = sent.index[sent["cluster"] == c]
    sims = emb[idx] @ cen / np.linalg.norm(cen)
    top = idx[np.argsort(-sims)[:5]]
    print(f"\n=== cluster {c}  (n={len(idx):,}) ===")
    for i in top:
        print("  -", sent.at[i, "sentence"][:200])

In [ ]:
sent.drop(columns=[c for c in sent.columns if c == "sentences"]).to_csv("sentence_clusters.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

xy = PCA(n_components=2, random_state=0).fit_transform(emb)
plt.figure(figsize=(9, 7))
plt.scatter(xy[:, 0], xy[:, 1], c=sent["cluster"], s=4, cmap="tab20")
plt.title("Sentence embeddings (PCA), coloured by cluster")
plt.show()